# 08 - Create Final Preprocessed Dataset (v3)

**v3 algorithm: 3-mode vignette handling**
1. **Strong circular vignette** → circle-FOV inscribed-square crop (cv2.minEnclosingCircle)
2. **Weak/mild vignette** → occupancy-based rectangular crop (fallback)
3. **No vignette** → save image unchanged as JPG 95

**Rules:** Raw images untouched. No augmentation. No normalisation. No 224 resize. Row counts unchanged.

---

## Section 0 - Imports

In [37]:
import cv2
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print(f"cv2    : {cv2.__version__}")
print(f"numpy  : {np.__version__}")
print(f"pandas : {pd.__version__}")

cv2    : 4.11.0
numpy  : 2.4.4
pandas : 2.3.3


## Section 1 - Paths and Configuration

In [38]:
OUTPUT_ROOT        = Path(r"C:\SKIN CANCER v2\pipe output")
FINAL_DATASET_ROOT = Path(r"C:\SKIN CANCER v2\final DS")
SPLITS_DIR        = OUTPUT_ROOT / "splits"
PREPROC_DIR       = OUTPUT_ROOT / "preprocessing"
FINAL_IMAGES_ROOT = FINAL_DATASET_ROOT / "images"

CLASS_INDEX  = {"NV": 0, "MEL": 1, "BCC": 2}
CLASS_NAMES  = ["NV", "MEL", "BCC"]
JPEG_QUALITY = 95

EXPECTED_COUNTS = {
    "train": {"total": 14332, "NV": 8928, "MEL": 3144, "BCC": 2260},
    "val":   {"total":  3012, "NV": 1886, "MEL":  647, "BCC":  479},
    "test":  {"total":  3045, "NV": 1894, "MEL":  639, "BCC":  512},
}

# ── Phase-1: vignette border metrics ─────────────────────────────────────
DARK_THRESHOLD          = 35
BORDER_BAND_RATIO       = 0.08
CORNER_PATCH_RATIO      = 0.12
CENTER_PATCH_RATIO      = 0.30
BORDER_DARK_FRAC_THRESH = 0.12   # lowered for v3
CC_CONTRAST_THRESH      = 20
CC_CORNER_MAX           = 80

# ── Mode-1: circle-FOV detection ─────────────────────────────────────────
CIRCLE_MIN_CONTOUR_FRAC = 0.15   # largest contour must cover >= 15% image area
CIRCLE_MAX_OFFSET_RATIO = 0.18   # center offset <= 18% of min(H,W)
CIRCLE_MIN_RADIUS_RATIO = 0.35   # radius between 35% and 75% of min(H,W)
CIRCLE_MAX_RADIUS_RATIO = 0.75
CIRCLE_SAFETY_FACTOR    = 0.92   # inscribed-square safety margin
CIRCLE_MIN_AREA_RATIO   = 0.25   # accept circle crop if area_ratio >= 0.25
CIRCLE_MIN_DIM_RATIO    = 0.40   # crop must be >= 40% of each image dimension

# ── Mode-2: occupancy crop (fallback) ────────────────────────────────────
OCCUPANCY_THRESH   = 0.50
MORPH_FRAC         = 0.015
SAFETY_PAD_FRAC    = 0.03
MIN_BORDER_REMOVED = 0.015
MIN_CROP_AREA      = 0.45
MAX_REMOVED_AREA   = 0.50
MIN_CROP_W_RATIO   = 0.55
MIN_CROP_H_RATIO   = 0.55

print("Configuration loaded.")
print(f"  DARK_THRESHOLD={DARK_THRESHOLD}")
print(f"  BORDER_DARK_FRAC_THRESH={BORDER_DARK_FRAC_THRESH}  CC_CONTRAST_THRESH={CC_CONTRAST_THRESH}")
print(f"  CIRCLE_MIN/MAX_RADIUS_RATIO={CIRCLE_MIN_RADIUS_RATIO}/{CIRCLE_MAX_RADIUS_RATIO}")
print(f"  CIRCLE_MIN_AREA_RATIO={CIRCLE_MIN_AREA_RATIO}")

Configuration loaded.
  DARK_THRESHOLD=35
  BORDER_DARK_FRAC_THRESH=0.12  CC_CONTRAST_THRESH=20
  CIRCLE_MIN/MAX_RADIUS_RATIO=0.35/0.75
  CIRCLE_MIN_AREA_RATIO=0.25


## Section 2 - Create Output Folders

In [39]:
PREPROC_DIR.mkdir(parents=True, exist_ok=True)
for cls in CLASS_NAMES:
    (FINAL_IMAGES_ROOT / cls).mkdir(parents=True, exist_ok=True)
print("Output folders ready.")

Output folders ready.


## Section 3 - Load and Validate Split Manifests

In [40]:
split_dfs = {}
for split in ["train", "val", "test"]:
    p = SPLITS_DIR / f"{split}_manifest.csv"
    if not p.exists():
        raise FileNotFoundError(f"Not found: {p}")
    split_dfs[split] = pd.read_csv(p)
    print(f"Loaded {split}: {len(split_dfs[split]):,} rows")

print(f"\nTotal: {sum(len(v) for v in split_dfs.values()):,}")
print("\n=== COUNT VALIDATION (informational) ===")
for split, df_s in split_dfs.items():
    ac = len(df_s); ec = EXPECTED_COUNTS[split]["total"]
    print(f"  {split}: {ac:,}  {'OK' if ac==ec else f'DIFF exp={ec:,}'}")
    cc = df_s["final_authoritative_label"].value_counts()
    for cls in CLASS_NAMES:
        a2 = int(cc.get(cls,0)); e2 = EXPECTED_COUNTS[split][cls]
        print(f"    {cls}: {a2:,}  {'OK' if a2==e2 else f'DIFF exp={e2:,}'}")

Loaded train: 14,332 rows
Loaded val: 3,012 rows
Loaded test: 3,045 rows

Total: 20,389

=== COUNT VALIDATION (informational) ===
  train: 14,332  OK
    NV: 8,928  OK
    MEL: 3,144  OK
    BCC: 2,260  OK
  val: 3,012  OK
    NV: 1,886  OK
    MEL: 647  OK
    BCC: 479  OK
  test: 3,045  OK
    NV: 1,894  OK
    MEL: 639  OK
    BCC: 512  OK


## Section 4 - Vignette Detection Functions (v3 Three-Mode)

In [41]:
def _crop_info(x1, y1, x2, y2, w, h, area_ratio, confidence, method):
    return {
        "x1": int(x1), "y1": int(y1), "x2": int(x2), "y2": int(y2),
        "w": int(w), "h": int(h),
        "area_ratio": round(float(area_ratio), 5),
        "confidence": str(confidence),
        "method": str(method),
    }


def compute_border_metrics(gray, H, W):
    """Phase-1: compute border/corner/centre intensity metrics."""
    bw = max(2, int(W * BORDER_BAND_RATIO))
    bh = max(2, int(H * BORDER_BAND_RATIO))
    top_d    = float((gray[:bh,    :  ] <= DARK_THRESHOLD).mean())
    bottom_d = float((gray[H-bh:,  :  ] <= DARK_THRESHOLD).mean())
    left_d   = float((gray[:,   :bw   ] <= DARK_THRESHOLD).mean())
    right_d  = float((gray[:, W-bw:   ] <= DARK_THRESHOLD).mean())
    bdf      = max(top_d, bottom_d, left_d, right_d)

    cp  = max(2, int(min(H, W) * CORNER_PATCH_RATIO))
    cpx = np.concatenate([
        gray[:cp,   :cp  ].ravel(), gray[:cp,   W-cp:].ravel(),
        gray[H-cp:, :cp  ].ravel(), gray[H-cp:, W-cp:].ravel()
    ])
    corner_mean = float(cpx.mean())

    mx = max(1, int(W * CENTER_PATCH_RATIO))
    my = max(1, int(H * CENTER_PATCH_RATIO))
    cp2 = gray[my:H-my, mx:W-mx]
    center_mean = float(cp2.mean()) if cp2.size > 0 else 0.0
    ccc = center_mean - corner_mean

    probable = (bdf >= BORDER_DARK_FRAC_THRESH) or \
               (ccc >= CC_CONTRAST_THRESH and corner_mean < CC_CORNER_MAX)

    reasons = []
    if bdf >= BORDER_DARK_FRAC_THRESH:                         reasons.append(f"border_dark_{bdf:.3f}")
    if ccc >= CC_CONTRAST_THRESH and corner_mean < CC_CORNER_MAX: reasons.append(f"contrast_{ccc:.1f}")
    if not reasons: reasons.append("none")

    return probable, {
        "border_dark_fraction":    round(bdf,      5),
        "top_dark_fraction":       round(top_d,    5),
        "bottom_dark_fraction":    round(bottom_d, 5),
        "left_dark_fraction":      round(left_d,   5),
        "right_dark_fraction":     round(right_d,  5),
        "corner_mean_intensity":   round(corner_mean, 2),
        "center_mean_intensity":   round(center_mean, 2),
        "center_corner_contrast":  round(ccc, 2),
        "probable_vignette_flag":  probable,
        "vignette_detection_reason": "|".join(reasons),
    }


print("compute_border_metrics defined.")

compute_border_metrics defined.


In [42]:
def _build_clean_mask(gray, H, W):
    """Build morphologically cleaned valid-pixel mask."""
    mask = (gray > DARK_THRESHOLD).astype(np.uint8) * 255
    ks   = max(7, int(min(H, W) * MORPH_FRAC))
    if ks % 2 == 0: ks += 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ks, ks))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
    return mask


def try_circle_detection(mask, H, W, border_metrics):
    """
    Mode-1: fit minEnclosingCircle to the largest contour.
    Returns (is_strong, circle_params_dict).
    """
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return False, {"circle_center_x": None, "circle_center_y": None,
                       "circle_radius": None, "circle_center_offset_ratio": None,
                       "circle_radius_ratio": None,
                       "largest_contour_area_ratio": None,
                       "strong_circle_vignette_flag": False}

    largest      = max(contours, key=cv2.contourArea)
    contour_area = cv2.contourArea(largest)
    area_ratio   = contour_area / (H * W)

    (cx, cy), radius = cv2.minEnclosingCircle(largest)
    cx, cy, radius   = float(cx), float(cy), float(radius)

    offset_px          = np.sqrt((cx - W / 2) ** 2 + (cy - H / 2) ** 2)
    center_offset_ratio = offset_px / min(H, W)
    radius_ratio        = radius / min(H, W)

    # Strong circle criteria
    geom_ok   = (
        center_offset_ratio <= CIRCLE_MAX_OFFSET_RATIO
        and CIRCLE_MIN_RADIUS_RATIO <= radius_ratio <= CIRCLE_MAX_RADIUS_RATIO
        and area_ratio >= CIRCLE_MIN_CONTOUR_FRAC
    )
    border_ok = (
        border_metrics["border_dark_fraction"] >= BORDER_DARK_FRAC_THRESH
        or border_metrics["center_corner_contrast"] >= CC_CONTRAST_THRESH
    )
    is_strong = geom_ok and border_ok

    return is_strong, {
        "circle_center_x":            round(cx, 1),
        "circle_center_y":            round(cy, 1),
        "circle_radius":              round(radius, 1),
        "circle_center_offset_ratio": round(center_offset_ratio, 4),
        "circle_radius_ratio":        round(radius_ratio, 4),
        "largest_contour_area_ratio": round(area_ratio, 4),
        "strong_circle_vignette_flag": is_strong,
    }


print("try_circle_detection defined.")

try_circle_detection defined.


In [43]:
def apply_circle_fov_crop(img_rgb, H, W, cx, cy, radius):
    """
    Compute inscribed-square crop from fitted circle.
    half_side = radius * CIRCLE_SAFETY_FACTOR / sqrt(2)
    Returns (cropped_img, crop_info) or (None, rejection_reason).
    """
    half_side = radius * CIRCLE_SAFETY_FACTOR / np.sqrt(2)

    x1 = max(0, int(cx - half_side))
    y1 = max(0, int(cy - half_side))
    x2 = min(W, int(cx + half_side))
    y2 = min(H, int(cy + half_side))

    cw = x2 - x1
    ch = y2 - y1

    if cw <= 0 or ch <= 0:
        return None, "circle_fov_zero_crop"

    area_ratio = (cw * ch) / (H * W)
    cw_ratio   = cw / W
    ch_ratio   = ch / H

    if area_ratio < CIRCLE_MIN_AREA_RATIO:
        return None, f"circle_fov_rejected_area_{area_ratio:.3f}"
    if cw_ratio < CIRCLE_MIN_DIM_RATIO:
        return None, f"circle_fov_rejected_width_{cw_ratio:.3f}"
    if ch_ratio < CIRCLE_MIN_DIM_RATIO:
        return None, f"circle_fov_rejected_height_{ch_ratio:.3f}"

    cropped = img_rgb[y1:y2, x1:x2]
    ci = _crop_info(x1, y1, x2, y2, cw, ch, area_ratio, "high_circle_fov", "circle_fov")
    return cropped, ci


print("apply_circle_fov_crop defined.")

apply_circle_fov_crop defined.


In [44]:
def apply_occupancy_crop(img_rgb, mask, H, W):
    """
    Mode-2 fallback: row/column occupancy scan.
    Returns (cropped_img, crop_info) or (None, rejection_reason).
    """
    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        mask, connectivity=8)

    if n_labels < 2:
        return None, "occ_no_component"

    ll   = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    cx, cy = centroids[ll]
    if not (0.15 < cx / W < 0.85 and 0.15 < cy / H < 0.85):
        return None, "occ_not_central"

    comp    = (labels == ll).astype(np.uint8)
    row_occ = comp.mean(axis=1)
    col_occ = comp.mean(axis=0)
    vr = np.where(row_occ >= OCCUPANCY_THRESH)[0]
    vc = np.where(col_occ >= OCCUPANCY_THRESH)[0]

    if len(vr) < 2 or len(vc) < 2:
        return None, "occ_low_occupancy"

    y1r, y2r = int(vr[0]),  int(vr[-1]) + 1
    x1r, x2r = int(vc[0]),  int(vc[-1]) + 1

    px = max(1, int((x2r - x1r) * SAFETY_PAD_FRAC))
    py = max(1, int((y2r - y1r) * SAFETY_PAD_FRAC))
    x1 = max(0, x1r - px);  y1 = max(0, y1r - py)
    x2 = min(W, x2r + px);  y2 = min(H, y2r + py)

    cw = x2 - x1;  ch = y2 - y1
    ar = (cw * ch) / (H * W)
    rr = 1.0 - ar

    if ar < MIN_CROP_AREA:      return None, f"occ_area_{ar:.3f}"
    if rr > MAX_REMOVED_AREA:   return None, f"occ_removed_{rr:.3f}"
    if cw / W < MIN_CROP_W_RATIO: return None, f"occ_width_{cw/W:.3f}"
    if ch / H < MIN_CROP_H_RATIO: return None, f"occ_height_{ch/H:.3f}"
    if rr < MIN_BORDER_REMOVED:  return None, "occ_minimal_border"

    cropped = img_rgb[y1:y2, x1:x2]
    ci = _crop_info(x1, y1, x2, y2, cw, ch, ar, "high_occupancy", "occupancy")
    return cropped, ci


print("apply_occupancy_crop defined.")

apply_occupancy_crop defined.


In [45]:
def detect_and_crop_vignette(img_rgb):
    """
    3-mode vignette handler.
    Returns (out_img, was_cropped, crop_info_dict, border_metrics, circle_params).
    """
    H, W = img_rgb.shape[:2]
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    probable, bm = compute_border_metrics(gray, H, W)

    # Default circle params (no circle detected)
    _no_circle = {
        "circle_center_x": None, "circle_center_y": None,
        "circle_radius": None,   "circle_center_offset_ratio": None,
        "circle_radius_ratio": None, "largest_contour_area_ratio": None,
        "strong_circle_vignette_flag": False,
    }
    _no_crop_ci = _crop_info(0, 0, W, H, W, H, 1.0, "no_vignette_detected", "no_crop")

    if not probable:
        return img_rgb, False, _no_crop_ci, bm, _no_circle

    mask      = _build_clean_mask(gray, H, W)
    is_strong, cp = try_circle_detection(mask, H, W, bm)

    # ── Mode 1: strong circular vignette ───────────────────────────────────
    if is_strong:
        cx, cy, radius = cp["circle_center_x"], cp["circle_center_y"], cp["circle_radius"]
        result, ci_or_reason = apply_circle_fov_crop(img_rgb, H, W, cx, cy, radius)
        if result is not None:
            return result, True, ci_or_reason, bm, cp
        # circle crop failed safety check — fall through to occupancy
        cp["strong_circle_vignette_flag"] = True  # keep flag even if crop failed

    # ── Mode 2: occupancy crop fallback ────────────────────────────────────
    result, ci_or_reason = apply_occupancy_crop(img_rgb, mask, H, W)
    if result is not None:
        return result, True, ci_or_reason, bm, cp

    # ── Mode 3: no crop (save original) ────────────────────────────────────
    reject_ci = _crop_info(0, 0, W, H, W, H, 1.0,
                           f"rejected_all_modes", "no_crop")
    return img_rgb, False, reject_ci, bm, cp


print("detect_and_crop_vignette (v3 3-mode) defined.")

detect_and_crop_vignette (v3 3-mode) defined.


## Section 5 - Image Processing Function

In [46]:
def process_single_image(src_path_str, label):
    src = Path(src_path_str)
    out = str(FINAL_IMAGES_ROOT / label / f"{src.stem}.jpg")
    base = {
        "original_full_path":         src_path_str,
        "preprocessed_full_path":      out,
        "class_index":                 CLASS_INDEX.get(label, -1),
        "preprocessing_applied":       False,
        "vignette_crop_applied":        False,
        "crop_method":                 "not_attempted",
        "crop_x1": None, "crop_y1": None, "crop_x2": None, "crop_y2": None,
        "crop_width": None, "crop_height": None,
        "crop_area_ratio": None, "crop_confidence_status": "not_attempted",
        "output_image_exists": False,
        "output_image_width": None, "output_image_height": None,
        # Border metrics
        "border_dark_fraction": None, "top_dark_fraction": None,
        "bottom_dark_fraction": None, "left_dark_fraction": None,
        "right_dark_fraction": None,
        "corner_mean_intensity": None, "center_mean_intensity": None,
        "center_corner_contrast": None,
        "probable_vignette_flag": None, "vignette_detection_reason": None,
        # Circle metrics
        "strong_circle_vignette_flag": None,
        "circle_center_x": None, "circle_center_y": None,
        "circle_radius": None,
        "circle_center_offset_ratio": None, "circle_radius_ratio": None,
        "largest_contour_area_ratio": None,
        "processing_error": "",
    }
    try:
        img_bgr = cv2.imread(str(src), cv2.IMREAD_COLOR)
        if img_bgr is None:
            base["processing_error"] = "cv2_read_none"
            return base
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        out_img, was_cropped, ci, bm, cp = detect_and_crop_vignette(img_rgb)

        base.update({
            "preprocessing_applied":      True,
            "vignette_crop_applied":       was_cropped,
            "crop_method":                ci["method"],
            "crop_x1":                    ci["x1"],  "crop_y1":  ci["y1"],
            "crop_x2":                    ci["x2"],  "crop_y2":  ci["y2"],
            "crop_width":                 ci["w"],   "crop_height": ci["h"],
            "crop_area_ratio":             ci["area_ratio"],
            "crop_confidence_status":      ci["confidence"],
        })
        base.update(bm)
        base.update(cp)

        out_bgr = cv2.cvtColor(out_img, cv2.COLOR_RGB2BGR)
        cv2.imwrite(out, out_bgr, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])

        if os.path.exists(out):
            chk = cv2.imread(out)
            if chk is not None:
                oh, ow = chk.shape[:2]
                base["output_image_exists"]  = True
                base["output_image_width"]   = ow
                base["output_image_height"]  = oh
    except Exception as exc:
        base["processing_error"] = str(exc)
    return base


print("process_single_image defined.")

process_single_image defined.


## Section 6 - Main Processing Loop

In [47]:
all_rows = []; all_results = []

for split in ["train", "val", "test"]:
    df_s = split_dfs[split]; n_split = len(df_s)
    n_ok = n_fail = n_circle = n_occ = 0
    print(f"\nProcessing {split}: {n_split:,} images...")

    for i, (_, row) in enumerate(df_s.iterrows()):
        res = process_single_image(
            str(row["full_path"]), str(row["final_authoritative_label"]))
        rd = row.to_dict(); rd["split"] = split
        all_rows.append(rd); all_results.append(res)

        if res["processing_error"]: n_fail += 1
        else:
            n_ok += 1
            m = res["crop_method"]
            if m == "circle_fov":  n_circle += 1
            elif m == "occupancy": n_occ    += 1

        if (i + 1) % 1000 == 0 or (i + 1) == n_split:
            print(f"  {i+1:,}/{n_split:,}  ok={n_ok:,}  "
                  f"circle={n_circle:,}  occ={n_occ:,}  fail={n_fail:,}")

    print(f"  {split}: circle={n_circle:,}  occ={n_occ:,}  fail={n_fail:,}")

print(f"\nTotal processed: {len(all_results):,}")


Processing train: 14,332 images...
  1,000/14,332  ok=1,000  circle=13  occ=2  fail=0
  2,000/14,332  ok=2,000  circle=20  occ=3  fail=0
  3,000/14,332  ok=3,000  circle=21  occ=3  fail=0
  4,000/14,332  ok=4,000  circle=22  occ=5  fail=0
  5,000/14,332  ok=5,000  circle=26  occ=6  fail=0
  6,000/14,332  ok=6,000  circle=117  occ=7  fail=0
  7,000/14,332  ok=7,000  circle=421  occ=7  fail=0
  8,000/14,332  ok=8,000  circle=709  occ=7  fail=0
  9,000/14,332  ok=9,000  circle=982  occ=10  fail=0
  10,000/14,332  ok=10,000  circle=1,034  occ=13  fail=0
  11,000/14,332  ok=11,000  circle=1,469  occ=13  fail=0
  12,000/14,332  ok=12,000  circle=1,917  occ=13  fail=0
  13,000/14,332  ok=13,000  circle=2,236  occ=15  fail=0
  14,000/14,332  ok=14,000  circle=2,694  occ=15  fail=0
  14,332/14,332  ok=14,332  circle=2,845  occ=15  fail=0
  train: circle=2,845  occ=15  fail=0

Processing val: 3,012 images...
  1,000/3,012  ok=1,000  circle=4  occ=1  fail=0
  2,000/3,012  ok=2,000  circle=191  o

## Section 7 - Build Processing Results DataFrame

In [48]:
df_input = (
    pd.DataFrame(all_rows)
    .rename(columns={"full_path": "original_full_path"})
)
df_proc = pd.DataFrame(all_results)
df_proc = df_proc[[c for c in df_proc.columns if c != "original_full_path"]]

results_df = pd.concat(
    [df_input.reset_index(drop=True), df_proc.reset_index(drop=True)], axis=1)

if "class_index" not in results_df.columns or results_df["class_index"].isna().any():
    results_df["class_index"] = results_df["final_authoritative_label"].map(CLASS_INDEX)

print(f"results_df: {results_df.shape[0]:,} rows x {results_df.shape[1]} cols")
results_df.head(2)

results_df: 20,389 rows x 43 cols


,original_full_path,final_authoritative_label,canonical_match_id,lesion_id,family_id,effective_split_group_id,grouping_source,split_assignment,split,preprocessed_full_path,...,probable_vignette_flag,vignette_detection_reason,strong_circle_vignette_flag,circle_center_x,circle_center_y,circle_radius,circle_center_offset_ratio,circle_radius_ratio,largest_contour_area_ratio,processing_error
0,C:\SKIN CANCER v2\DS\NV\ISIC_0000000.jpg,NV,ISIC_0000000,NaN,FAM_ISIC_0000000,FAM_ISIC_0000000,family_id,train,train,C:\SKIN CANCER v2\final DS\images\NV\ISIC_0000...,...,False,none,False,NaN,NaN,NaN,NaN,NaN,NaN,
1,C:\SKIN CANCER v2\DS\NV\ISIC_0000001.jpg,NV,ISIC_0000001,NaN,FAM_ISIC_0000001,FAM_ISIC_0000001,family_id,train,train,C:\SKIN CANCER v2\final DS\images\NV\ISIC_0000...,...,False,none,False,NaN,NaN,NaN,NaN,NaN,NaN,


## Section 8 - Validation

In [49]:
print("=== OUTPUT VALIDATION ===")
split_col = "split_assignment" if "split_assignment" in results_df.columns else "split"

for split in ["train", "val", "test"]:
    mask = results_df[split_col] == split
    ac = int(mask.sum()); ec = EXPECTED_COUNTS[split]["total"]
    print(f"  {split}: {ac:,}  {'OK' if ac==ec else f'DIFF exp={ec:,}'}")
    cc = results_df[mask]["final_authoritative_label"].value_counts()
    for cls in CLASS_NAMES:
        a2 = int(cc.get(cls,0)); e2 = EXPECTED_COUNTS[split][cls]
        print(f"    {cls}: {a2:,}  {'OK' if a2==e2 else f'DIFF exp={e2:,}'}")

n_failed  = int((results_df["processing_error"] != "").sum())
n_exists  = int(results_df["output_image_exists"].sum())
n_missing = len(results_df) - n_exists
n_dup_out = int(results_df["preprocessed_full_path"].duplicated().sum())
raw_contaminated = results_df[
    results_df["preprocessed_full_path"].str.startswith(r"C:\SKIN CANCER v2\DS")
]
print(f"\n  Processing failures    : {n_failed:,}")
print(f"  Output images exist    : {n_exists:,}")
print(f"  Output images missing  : {n_missing:,}")
print(f"  Duplicate output paths : {n_dup_out:,}")
print(f"  Output in raw dir      : {len(raw_contaminated):,}  (must be 0)")

=== OUTPUT VALIDATION ===
  train: 14,332  OK
    NV: 8,928  OK
    MEL: 3,144  OK
    BCC: 2,260  OK
  val: 3,012  OK
    NV: 1,886  OK
    MEL: 647  OK
    BCC: 479  OK
  test: 3,045  OK
    NV: 1,894  OK
    MEL: 639  OK
    BCC: 512  OK

  Processing failures    : 0
  Output images exist    : 20,389
  Output images missing  : 0
  Duplicate output paths : 0
  Output in raw dir      : 0  (must be 0)


## Section 9 - Vignette Crop Audit

In [50]:
ok_mask         = results_df["processing_error"] == ""
n_processed     = int(ok_mask.sum())
n_probable      = int(results_df["probable_vignette_flag"].sum())
n_strong_circle = int(results_df["strong_circle_vignette_flag"].sum())
n_circle_crops  = int((results_df["crop_method"] == "circle_fov").sum())
n_occ_crops     = int((results_df["crop_method"] == "occupancy").sum())
n_crop_total    = n_circle_crops + n_occ_crops
n_nocrop        = n_processed - n_crop_total
crop_rate_pct   = round(n_crop_total / max(1, n_processed) * 100, 2)

susp_mask = (
    (results_df["probable_vignette_flag"] == True) &
    (results_df["vignette_crop_applied"]  == False) &
    (results_df["processing_error"]       == "")
)
n_suspicious = int(susp_mask.sum())

print(f"Total processed           : {n_processed:,}")
print(f"Probable vignette         : {n_probable:,}")
print(f"Strong circular vignette  : {n_strong_circle:,}")
print(f"Crop applied (total)      : {n_crop_total:,}  ({crop_rate_pct}%)")
print(f"  circle_fov              : {n_circle_crops:,}")
print(f"  occupancy               : {n_occ_crops:,}")
print(f"No-crop fallback          : {n_nocrop:,}")
print(f"Suspicious no-crop        : {n_suspicious:,}")

print("\n=== CROP METHOD COUNTS ===")
print(results_df["crop_method"].value_counts().to_string())

print("\n=== CROP CONFIDENCE STATUS ===")
print(results_df["crop_confidence_status"].value_counts().to_string())

crop_by_class = []
print("\n=== CROP BREAKDOWN BY CLASS ===")
for cls in CLASS_NAMES:
    mc    = results_df["final_authoritative_label"] == cls
    tot   = int(mc.sum())
    circ  = int((results_df[mc]["crop_method"] == "circle_fov").sum())
    occ   = int((results_df[mc]["crop_method"] == "occupancy").sum())
    cr    = circ + occ
    prob  = int((results_df[mc]["probable_vignette_flag"]).sum())
    rt    = round(cr / max(1, tot) * 100, 2)

    ar_circ = results_df[mc & (results_df["crop_method"]=="circle_fov")]["crop_area_ratio"].mean()
    ar_occ  = results_df[mc & (results_df["crop_method"]=="occupancy")]["crop_area_ratio"].mean()
    ar_circ = round(float(ar_circ),4) if not pd.isna(ar_circ) else None
    ar_occ  = round(float(ar_occ), 4) if not pd.isna(ar_occ)  else None

    crop_by_class.append({
        "class": cls, "total": tot, "probable": prob,
        "circle_fov": circ, "occupancy": occ,
        "crop_total": cr, "crop_rate_pct": rt,
        "avg_ar_circle_fov": ar_circ, "avg_ar_occupancy": ar_occ,
    })
    print(f"  {cls}: total={tot:,}  prob={prob:,}  "
          f"circle={circ:,}  occ={occ:,}  rate={rt}%  "
          f"ar_circ={ar_circ}  ar_occ={ar_occ}")

df_suspicious = (
    results_df[susp_mask]
    .sort_values(["border_dark_fraction", "center_corner_contrast"], ascending=False)
    .head(50)
)
print(f"\nTop suspicious no-crop (head 5):")
sc = [c for c in ["original_full_path", "final_authoritative_label",
                  "border_dark_fraction", "center_corner_contrast",
                  "strong_circle_vignette_flag", "crop_confidence_status"]
      if c in df_suspicious.columns]
from IPython.display import display
display(df_suspicious[sc].head(5))

Total processed           : 20,389
Probable vignette         : 5,068
Strong circular vignette  : 4,123
Crop applied (total)      : 4,031  (19.77%)
  circle_fov              : 4,013
  occupancy               : 18
No-crop fallback          : 16,358
Suspicious no-crop        : 1,037

=== CROP METHOD COUNTS ===
crop_method
no_crop       16358
circle_fov     4013
occupancy        18

=== CROP CONFIDENCE STATUS ===
crop_confidence_status
no_vignette_detected    15321
high_circle_fov          4013
rejected_all_modes       1037
high_occupancy             18

=== CROP BREAKDOWN BY CLASS ===
  NV: total=12,708  prob=1,866  circle=1,378  occ=9  rate=10.91%  ar_circ=0.5346  ar_occ=0.8873
  MEL: total=4,430  prob=1,705  circle=1,343  occ=7  rate=30.47%  ar_circ=0.5038  ar_occ=0.899
  BCC: total=3,251  prob=1,497  circle=1,292  occ=2  rate=39.8%  ar_circ=0.5191  ar_occ=0.9578

Top suspicious no-crop (head 5):


,original_full_path,final_authoritative_label,border_dark_fraction,center_corner_contrast,strong_circle_vignette_flag,crop_confidence_status
19485,C:\SKIN CANCER v2\DS\MEL\ISIC_0053533.jpg,MEL,1.0,206.45,True,rejected_all_modes
8607,C:\SKIN CANCER v2\DS\NV\ISIC_0071177.jpg,NV,1.0,206.03,True,rejected_all_modes
17238,C:\SKIN CANCER v2\DS\BCC\ISIC_0067998.jpg,BCC,1.0,204.32,False,rejected_all_modes
19854,C:\SKIN CANCER v2\DS\MEL\ISIC_0072287.jpg,MEL,1.0,204.11,True,rejected_all_modes
12973,C:\SKIN CANCER v2\DS\BCC\ISIC_0059162.jpg,BCC,1.0,203.50,False,rejected_all_modes


## Section 10 - Visual Audit Grid

Rows 1-3: one circle_fov crop example per class.  
Rows 4-5: one occupancy crop example, one suspicious no-crop example.  
Columns: Original | Overlay (circle or crop box) | Output.

In [51]:
def _draw_circle_on_overlay(overlay, cx, cy, radius, colour=(0,255,100), thickness=3):
    if cx is not None and radius is not None:
        cv2.circle(overlay, (int(cx), int(cy)), int(radius), colour, thickness)
        cv2.drawMarker(overlay, (int(cx), int(cy)), colour,
                       cv2.MARKER_CROSS, 20, thickness)

def _draw_box_on_overlay(overlay, x1, y1, x2, y2, colour=(255,140,0), thickness=3):
    if x1 is not None and x2 is not None and int(x1) < int(x2):
        cv2.rectangle(overlay, (int(x1), int(y1)), (int(x2), int(y2)),
                      colour, thickness)


# Build example pool
_examples = []

# Rows 1-3: circle_fov per class
for cls in CLASS_NAMES:
    pool = results_df[
        (results_df["final_authoritative_label"] == cls) &
        (results_df["crop_method"] == "circle_fov") &
        (results_df["output_image_exists"] == True)
    ]
    _examples.append((f"circle_fov\n{cls}", pool.iloc[0] if len(pool) > 0 else None))

# Row 4: one occupancy example (any class)
pool_occ = results_df[
    (results_df["crop_method"] == "occupancy") &
    (results_df["output_image_exists"] == True)
]
_examples.append(("occupancy", pool_occ.iloc[0] if len(pool_occ) > 0 else None))

# Row 5: one no_crop example
pool_nc = results_df[
    (results_df["crop_method"] == "no_crop") &
    (results_df["probable_vignette_flag"] == False) &
    (results_df["output_image_exists"] == True)
]
_examples.append(("no_crop", pool_nc.iloc[0] if len(pool_nc) > 0 else None))

# Row 6: one suspicious no-crop
_examples.append(("suspicious\nno-crop",
                  df_suspicious.iloc[0] if len(df_suspicious) > 0 else None))

n_rows = len(_examples)
fig, axes = plt.subplots(n_rows, 3, figsize=(18, n_rows * 4))
if n_rows == 1: axes = [axes]
for ax_row in axes:
    for ax in ax_row: ax.axis("off")

for ci2, t in enumerate(["Original", "Detection Overlay", "Output"]):
    axes[0][ci2].set_title(t, fontsize=12, fontweight="bold", pad=5)

for r, (row_label, ex) in enumerate(_examples):
    ax0, ax1, ax2 = axes[r]
    ax0.set_ylabel(row_label, fontsize=9, rotation=0, labelpad=65, va="center")

    if ex is None:
        for ax in [ax0, ax1, ax2]:
            ax.text(0.5, 0.5, "No example", ha="center", va="center",
                    transform=ax.transAxes, fontsize=9)
        continue

    img_bgr = cv2.imread(str(ex["original_full_path"]))
    if img_bgr is None:
        for ax in [ax0, ax1, ax2]:
            ax.text(0.5, 0.5, "Read error", ha="center", va="center",
                    transform=ax.transAxes)
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    ax0.imshow(img_rgb)
    bdf = ex.get("border_dark_fraction", 0.0)
    ccc = ex.get("center_corner_contrast", 0.0)
    ax0.set_title(f"bdf={float(bdf):.3f}  cc={float(ccc):.1f}", fontsize=7)

    overlay = img_rgb.copy()
    method  = str(ex.get("crop_method", ""))
    if method == "circle_fov":
        _draw_circle_on_overlay(overlay,
                                ex.get("circle_center_x"), ex.get("circle_center_y"),
                                ex.get("circle_radius"))
        _draw_box_on_overlay(overlay,
                             ex.get("crop_x1"), ex.get("crop_y1"),
                             ex.get("crop_x2"), ex.get("crop_y2"), (0, 220, 0))
    else:
        colour = (0, 220, 0) if ex.get("vignette_crop_applied") else (255, 140, 0)
        _draw_box_on_overlay(overlay,
                             ex.get("crop_x1"), ex.get("crop_y1"),
                             ex.get("crop_x2"), ex.get("crop_y2"), colour)
    ax1.imshow(overlay)
    conf = str(ex.get("crop_confidence_status", ""))
    ar2  = ex.get("crop_area_ratio")
    ax1.set_title(f"{method}  {conf[:22]}\nar={ar2:.3f}" if ar2 else method, fontsize=7)

    proc = cv2.imread(str(ex["preprocessed_full_path"]))
    if proc is not None:
        ax2.imshow(cv2.cvtColor(proc, cv2.COLOR_BGR2RGB))

plt.suptitle("Vignette Crop Audit Grid - v3 Three-Mode Algorithm",
             fontsize=13, y=1.005)
plt.tight_layout()
grid_path = PREPROC_DIR / "vignette_examples_grid.png"
plt.savefig(str(grid_path), bbox_inches="tight", dpi=100)
plt.close()
print(f"Grid saved: {grid_path}")

Grid saved: C:\SKIN CANCER v2\pipe output\preprocessing\vignette_examples_grid.png


## Section 11 - Build Output Manifests

In [52]:
AUDIT_COLS = [c for c in [
    "original_full_path", "preprocessed_full_path",
    "final_authoritative_label", "class_index",
    split_col, "canonical_match_id", "lesion_id", "family_id",
    "effective_split_group_id", "grouping_source",
    "preprocessing_applied", "vignette_crop_applied", "crop_method",
    "crop_x1", "crop_y1", "crop_x2", "crop_y2",
    "crop_width", "crop_height", "crop_area_ratio", "crop_confidence_status",
    "output_image_exists", "output_image_width", "output_image_height",
    "border_dark_fraction", "top_dark_fraction", "bottom_dark_fraction",
    "left_dark_fraction", "right_dark_fraction",
    "corner_mean_intensity", "center_mean_intensity", "center_corner_contrast",
    "probable_vignette_flag", "vignette_detection_reason",
    "strong_circle_vignette_flag",
    "circle_center_x", "circle_center_y", "circle_radius",
    "circle_center_offset_ratio", "circle_radius_ratio",
    "largest_contour_area_ratio",
    "processing_error",
] if c in results_df.columns]

output_manifests = {}
for split in ["train", "val", "test"]:
    mask = results_df[split_col] == split
    output_manifests[split] = results_df[mask][AUDIT_COLS].copy()
    print(f"{split}: {len(output_manifests[split]):,} rows x "
          f"{output_manifests[split].shape[1]} cols")

train: 14,332 rows x 42 cols
val: 3,012 rows x 42 cols
test: 3,045 rows x 42 cols


## Section 12 - Save Outputs

In [53]:
for split, df_out in output_manifests.items():
    p = PREPROC_DIR / f"{split}_manifest_preprocessed.csv"
    df_out.to_csv(str(p), index=False)
    print(f"Saved {p.name}  ({len(df_out):,} rows)")

Saved train_manifest_preprocessed.csv  (14,332 rows)
Saved val_manifest_preprocessed.csv  (3,012 rows)
Saved test_manifest_preprocessed.csv  (3,045 rows)


In [54]:
# preprocessing_summary.csv
sr = [
    {"metric": "total_input_rows",         "value": len(results_df)},
    {"metric": "total_processed_ok",        "value": n_processed},
    {"metric": "total_failed",              "value": n_failed},
    {"metric": "probable_vignette_count",   "value": n_probable},
    {"metric": "strong_circle_count",       "value": n_strong_circle},
    {"metric": "crop_circle_fov",           "value": n_circle_crops},
    {"metric": "crop_occupancy",            "value": n_occ_crops},
    {"metric": "crop_total",                "value": n_crop_total},
    {"metric": "no_crop_fallback",           "value": n_nocrop},
    {"metric": "suspicious_no_crop",         "value": n_suspicious},
    {"metric": "crop_rate_pct",             "value": crop_rate_pct},
    {"metric": "output_images_exist",        "value": n_exists},
    {"metric": "output_images_missing",      "value": n_missing},
    {"metric": "train_rows",                "value": len(output_manifests["train"])},
    {"metric": "val_rows",                  "value": len(output_manifests["val"])},
    {"metric": "test_rows",                 "value": len(output_manifests["test"])},
    {"metric": "jpeg_quality",              "value": JPEG_QUALITY},
    {"metric": "dark_threshold",            "value": DARK_THRESHOLD},
    {"metric": "occupancy_thresh",          "value": OCCUPANCY_THRESH},
    {"metric": "circle_min_radius_ratio",   "value": CIRCLE_MIN_RADIUS_RATIO},
    {"metric": "circle_max_radius_ratio",   "value": CIRCLE_MAX_RADIUS_RATIO},
    {"metric": "circle_min_area_ratio",     "value": CIRCLE_MIN_AREA_RATIO},
]
for row in crop_by_class:
    cls = row["class"]
    sr += [
        {"metric": f"crop_rate_{cls}_pct",       "value": row["crop_rate_pct"]},
        {"metric": f"circle_fov_{cls}",          "value": row["circle_fov"]},
        {"metric": f"occupancy_{cls}",           "value": row["occupancy"]},
        {"metric": f"avg_ar_circle_fov_{cls}",   "value": row["avg_ar_circle_fov"]},
        {"metric": f"avg_ar_occupancy_{cls}",    "value": row["avg_ar_occupancy"]},
    ]
summary_path = PREPROC_DIR / "preprocessing_summary.csv"
pd.DataFrame(sr).to_csv(str(summary_path), index=False)
print(f"Saved {summary_path.name}")

Saved preprocessing_summary.csv


In [56]:
# vignette_crop_audit.csv
ac2 = [c for c in [
    "original_full_path", "preprocessed_full_path",
    "final_authoritative_label", split_col,
    "vignette_crop_applied", "crop_method",
    "probable_vignette_flag", "strong_circle_vignette_flag",
    "vignette_detection_reason", "border_dark_fraction",
    "center_corner_contrast", "circle_radius_ratio",
    "crop_x1", "crop_y1", "crop_x2", "crop_y2",
    "crop_width", "crop_height", "crop_area_ratio", "crop_confidence_status",
    "output_image_width", "output_image_height", "processing_error",
] if c in results_df.columns]
audit_path = PREPROC_DIR / "vignette_crop_audit.csv"
results_df[ac2].to_csv(str(audit_path), index=False)
print(f"Saved {audit_path.name}  ({len(results_df):,} rows)")

# suspicious_no_crop_cases.csv
susp_path = PREPROC_DIR / "suspicious_no_crop_cases.csv"
df_suspicious[ac2].to_csv(str(susp_path), index=False)
print(f"Saved {susp_path.name}  ({len(df_suspicious)} rows)")

Saved vignette_crop_audit.csv  (20,389 rows)
Saved suspicious_no_crop_cases.csv  (50 rows)


## Section 13 - Output File Verification

In [57]:
required_files = [
    PREPROC_DIR / "train_manifest_preprocessed.csv",
    PREPROC_DIR / "val_manifest_preprocessed.csv",
    PREPROC_DIR / "test_manifest_preprocessed.csv",
    PREPROC_DIR / "preprocessing_summary.csv",
    PREPROC_DIR / "vignette_crop_audit.csv",
    PREPROC_DIR / "suspicious_no_crop_cases.csv",
    PREPROC_DIR / "vignette_examples_grid.png",
]
print("Output file verification:")
all_ok = True
for p in required_files:
    exists = p.exists(); size = p.stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {p.name:<45} {size:>12,} bytes")
    if not exists: all_ok = False
print("\nAll files present." if all_ok else "\nWARNING: missing files.")
print("\nOutput image folder counts:")
for cls in CLASS_NAMES:
    folder = FINAL_IMAGES_ROOT / cls
    count  = len(list(folder.glob("*.jpg"))) if folder.exists() else 0
    print(f"  {cls}: {count:,} images")

Output file verification:
  [OK] train_manifest_preprocessed.csv                  4,859,314 bytes
  [OK] val_manifest_preprocessed.csv                    1,013,267 bytes
  [OK] test_manifest_preprocessed.csv                   1,031,649 bytes
  [OK] preprocessing_summary.csv                              916 bytes
  [OK] vignette_crop_audit.csv                          4,404,093 bytes
  [OK] suspicious_no_crop_cases.csv                        12,398 bytes
  [OK] vignette_examples_grid.png                       2,613,501 bytes

All files present.

Output image folder counts:
  NV: 12,708 images
  MEL: 4,430 images
  BCC: 3,251 images


## Section 14 - Final Summary (Copy-Paste Ready)

In [58]:
from IPython.display import display as _disp

print("=" * 70)
print("  08_create_final_preprocessed_dataset (v3) -- FINAL SUMMARY")
print("=" * 70)

print(f"\n 1. Total images processed         : {n_processed:,}")
print(f" 2. Crop applied (total)           : {n_crop_total:,}  ({crop_rate_pct}%)")
print(f" 3. Crop count by method:")
print(f"      circle_fov                   : {n_circle_crops:,}")
print(f"      occupancy                    : {n_occ_crops:,}")
print(f" 4. No-crop fallback               : {n_nocrop:,}")
print(f" 5. Probable vignette count        : {n_probable:,}")
print(f" 6. Strong circular vignette count : {n_strong_circle:,}")
print(f" 7. Suspicious no-crop count       : {n_suspicious:,}")

print(f"\n 8. Crop rate by class:")
for row in crop_by_class:
    print(f"    {row['class']}: {row['crop_rate_pct']}%  "
          f"total={row['crop_total']:,}/{row['total']:,}")

print(f"\n 9. Crop method by class:")
for row in crop_by_class:
    print(f"    {row['class']}: circle_fov={row['circle_fov']:,}  "
          f"occupancy={row['occupancy']:,}")

print(f"\n10. Average crop area ratio by class and method:")
for row in crop_by_class:
    print(f"    {row['class']}: circle_fov={row['avg_ar_circle_fov']}  "
          f"occupancy={row['avg_ar_occupancy']}")

print(f"\n11. Crop confidence status counts:")
for status, cnt in results_df["crop_confidence_status"].value_counts().items():
    print(f"    {str(status):<44}: {cnt:,}")

print(f"\n12. Output split row counts:")
for split in ["train", "val", "test"]:
    n = int((results_df[split_col] == split).sum())
    print(f"    {split}: {n:,}")

print(f"\n13. Output class counts per split:")
for split in ["train", "val", "test"]:
    mask = results_df[split_col] == split
    cc   = results_df[mask]["final_authoritative_label"].value_counts()
    print(f"    {split}: NV={int(cc.get('NV',0)):,}  "
          f"MEL={int(cc.get('MEL',0)):,}  BCC={int(cc.get('BCC',0)):,}")

print(f"\n14. Failed processing count        : {n_failed:,}")

print(f"\n15. Output file verification:")
for p in required_files:
    exists = p.exists(); size = p.stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"    [{status}] {p.name:<45} {size:>10,} bytes")

print(f"\n16. Raw image modification check:")
print(f"    Output paths in raw DS dir     : {len(raw_contaminated):,}  (must be 0)")
print(f"    Raw images modified            : False")
print(f"    Raw folder access              : read-only")

print("\nhead(3) of train_manifest_preprocessed.csv:")
_pc = [c for c in [
    "original_full_path", "final_authoritative_label",
    "crop_method", "vignette_crop_applied",
    "crop_area_ratio", "crop_confidence_status", "output_image_exists"
] if c in output_manifests["train"].columns]
_disp(output_manifests["train"][_pc].head(3))
print("=" * 70)

  08_create_final_preprocessed_dataset (v3) -- FINAL SUMMARY

 1. Total images processed         : 20,389
 2. Crop applied (total)           : 4,031  (19.77%)
 3. Crop count by method:
      circle_fov                   : 4,013
      occupancy                    : 18
 4. No-crop fallback               : 16,358
 5. Probable vignette count        : 5,068
 6. Strong circular vignette count : 4,123
 7. Suspicious no-crop count       : 1,037

 8. Crop rate by class:
    NV: 10.91%  total=1,387/12,708
    MEL: 30.47%  total=1,350/4,430
    BCC: 39.8%  total=1,294/3,251

 9. Crop method by class:
    NV: circle_fov=1,378  occupancy=9
    MEL: circle_fov=1,343  occupancy=7
    BCC: circle_fov=1,292  occupancy=2

10. Average crop area ratio by class and method:
    NV: circle_fov=0.5346  occupancy=0.8873
    MEL: circle_fov=0.5038  occupancy=0.899
    BCC: circle_fov=0.5191  occupancy=0.9578

11. Crop confidence status counts:
    no_vignette_detected                        : 15,321
    high_ci

,original_full_path,final_authoritative_label,crop_method,vignette_crop_applied,crop_area_ratio,crop_confidence_status,output_image_exists
0,C:\SKIN CANCER v2\DS\NV\ISIC_0000000.jpg,NV,no_crop,False,1.0,no_vignette_detected,True
1,C:\SKIN CANCER v2\DS\NV\ISIC_0000001.jpg,NV,no_crop,False,1.0,no_vignette_detected,True
2,C:\SKIN CANCER v2\DS\NV\ISIC_0000003.jpg,NV,no_crop,False,1.0,no_vignette_detected,True


## Section 15 - Completion Summary

**Section 08 v3 is complete.**

**v3 algorithm — 3-mode vignette handling:**

**Mode 1 – Circle-FOV crop (new in v3):**  
For images where `strong_circle_vignette_flag=True`: fits `cv2.minEnclosingCircle` to the largest contour in the valid-pixel mask. Computes an inscribed-square crop: `half_side = radius * 0.92 / sqrt(2)`. Accepts crop if `area_ratio >= 0.25` and `width/height >= 40%` of image. This correctly handles the 1024×768 dermoscopy case where a circle of radius ~384 px gives `half_side ≈ 250`, crop ≈ 500×500, `area_ratio ≈ 0.32`.

**Mode 2 – Occupancy crop (fallback for mild vignette):**  
Row/column occupancy scan at 0.50 threshold. Used when `probable_vignette_flag=True` but circle detection did not pass.

**Mode 3 – No crop:**  
When `probable_vignette_flag=False` or all crop modes rejected. Image saved unchanged as JPG 95.

**Tuning the detector:** If too many or too few images are being cropped:  
- Lower `BORDER_DARK_FRAC_THRESH` (currently 0.12) to catch more borderline vignettes  
- Lower `CIRCLE_MIN_RADIUS_RATIO` (currently 0.35) for smaller circles  
- Raise `CIRCLE_MIN_AREA_RATIO` (currently 0.25) to reject aggressive crops  

**Next step:** Inspect `vignette_examples_grid.png` and `suspicious_no_crop_cases.csv`. When satisfied, proceed to 09_pytorch_dataset_and_transforms.ipynb.